# 課題：チャットボットにキャラクターを設定しよう

システムプロンプトを使用してチャットボットにキャラクターを設定します。

In [ ]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
from pprint import pprint

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [ ]:
# メッセージを格納するリスト
# システムプロンプトでキャラクターを設定
messages=[
    {
        "role": "system",
        "content": "あなたは親切で礼儀正しい侍です。語尾に「〜でござる」「〜候（そうろう）」をつけて話し、武士らしい品格のある言葉遣いをします。常に敬意を持って接し、誠実に答えます。"
    }
]

while(True):
    # ユーザーからの質問を受付
    message = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if message.strip()=="":
        break
    print(f"質問:{message}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": message.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    # ただし、システムプロンプト（最初のメッセージ）は削除しない
    if len(messages) > 9:  # システムプロンプト1つ + ユーザーとアシスタントのメッセージ8つ
        del_message = messages.pop(1)  # インデックス1（システムプロンプトの次）から削除

    # APIへリクエスト
    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        stream=True,
    )

    # 言語モデルからの回答を表示
    response_message = ""
    for chunk in stream:
        if chunk.choices:
            next = chunk.choices[0].delta.content
            if next is not None:
                response_message += next
                print(next, end='', flush=True)

    # メッセージに言語モデルからの回答を追加
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")